In [1]:
# =============================================================================
# 02_G.ipynb — 셀 1: 환경 / 경로 / 유틸
# =============================================================================
# 목적: G팩터 (Short-term Reversal) 검증
# 가설: 단기 급락 종목이 평균회귀로 반등
# 논리: 과잉반응 → 복원, 유동성 충격 → 해소
# 선행연구: Jegadeesh (1990), Lehmann (1990)
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True

# ── SSOT 경로 ─────────────────────────────────────────────────────
QP2_ROOT = Path(r"C:\QP2")
DATA_DIR = QP2_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
META_DIR = DATA_DIR / "meta"

PATHS = {
    "px_wide":   INTERIM_DIR / "yahoo_adjclose_wide.parquet",
    "regime":    INTERIM_DIR / "regime_indicators_combined.parquet",
    "universe":  META_DIR / "sp500_universe.parquet",
}

# ── G-1 전용 파라미터 ─────────────────────────────────────────────
TOP_N = 30              # 포트폴리오 종목 수
COST_BP = 20            # 편도 거래비용 (bp)
REGIME_COL = "regime_v2"

# Reversal 파라미터
REV_LOOKBACK = 5        # 과거 N일 수익률 (formation)
REV_HOLD = 5            # 보유 기간 (일)

# ── 공통 유틸 함수 ────────────────────────────────────────────────
def calc_perf(ret_series):
    """수익률 시리즈 → CAGR, Sharpe, MaxDD"""
    r = ret_series.dropna()
    if len(r) < 20:
        return {"CAGR": np.nan, "Sharpe": np.nan, "MaxDD": np.nan, "N": len(r)}
    cum = (1 + r).cumprod()
    years = len(r) / 252
    cagr = cum.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
    sharpe = r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else 0
    dd = cum / cum.cummax() - 1
    maxdd = dd.min()
    return {"CAGR": cagr, "Sharpe": sharpe, "MaxDD": maxdd, "N": len(r)}

def calc_tstat(port_ret, bm_ret):
    """초과수익 t-stat"""
    excess = (port_ret - bm_ret).dropna()
    if len(excess) < 2:
        return np.nan
    return excess.mean() / (excess.std() / np.sqrt(len(excess)))

print("=" * 60)
print("02_G.ipynb — G팩터 (Short-term Reversal)")
print("=" * 60)
print(f"QP2_ROOT     : {QP2_ROOT}")
print(f"REV_LOOKBACK : {REV_LOOKBACK}일 (formation)")
print(f"REV_HOLD     : {REV_HOLD}일 (holding)")
print(f"TOP_N        : {TOP_N}")
print(f"REGIME_COL   : {REGIME_COL}")

02_G.ipynb — G팩터 (Short-term Reversal)
QP2_ROOT     : C:\QP2
REV_LOOKBACK : 5일 (formation)
REV_HOLD     : 5일 (holding)
TOP_N        : 30
REGIME_COL   : regime_v2


In [2]:
# =============================================================================
# 셀 2 — 데이터 로드
# =============================================================================
# 산출물: px_wide, ret_1d, regime_d, ew_ret_d, universe
# =============================================================================

# ── 주가 (일간) ───────────────────────────────────────────────────
px_wide = pd.read_parquet(PATHS["px_wide"])
if "date" in px_wide.columns:
    px_wide = px_wide.set_index("date")
px_wide.index = pd.to_datetime(px_wide.index)

# ── 유니버스 ──────────────────────────────────────────────────────
universe = pd.read_parquet(PATHS["universe"])
common_tk = sorted(set(universe["ticker_yahoo"]) & set(px_wide.columns))
px_wide = px_wide[common_tk]

# ── 일간 수익률 ───────────────────────────────────────────────────
ret_1d = px_wide.pct_change()

# ── EW 벤치마크 (일간) ────────────────────────────────────────────
ew_ret_d = ret_1d.mean(axis=1)
ew_ret_d.name = "EW"

# ── 레짐 (일간) ───────────────────────────────────────────────────
regime_raw = pd.read_parquet(PATHS["regime"])
if "date" in regime_raw.columns:
    regime_raw = regime_raw.set_index("date")
regime_raw.index = pd.to_datetime(regime_raw.index)

# 일간으로 리샘플 (ffill)
regime_d = regime_raw[[REGIME_COL]].reindex(ret_1d.index, method="ffill")

# ── 분석 기간 설정 (2013-06-19 이후) ──────────────────────────────
START_DATE = "2013-06-19"
px_wide = px_wide.loc[START_DATE:]
ret_1d = ret_1d.loc[START_DATE:]
ew_ret_d = ew_ret_d.loc[START_DATE:]
regime_d = regime_d.loc[START_DATE:]

print(f"px_wide  : {px_wide.shape[0]} days × {px_wide.shape[1]} tickers")
print(f"ret_1d   : {ret_1d.shape}")
print(f"기간     : {ret_1d.index.min().date()} ~ {ret_1d.index.max().date()}")
print(f"레짐 분포:\n{regime_d[REGIME_COL].value_counts().sort_index()}")

px_wide  : 3178 days × 503 tickers
ret_1d   : (3178, 503)
기간     : 2013-06-19 ~ 2026-02-05
레짐 분포:
regime_v2
0_Neutral            614
1_Crash               42
2_Recovery_Early      60
3_Contraction        147
4_Recovery_Late      879
5_Expansion         1334
6_Peak               102
Name: count, dtype: int64


In [3]:
# =============================================================================
# 셀 3 — Reversal 시그널 생성
# =============================================================================
# 목적: 과거 N일 수익률 기준 "급락 종목" 식별
# 산출물: rev_signal (일별 × 종목별 reversal z-score, 급락 = 양수)
# =============================================================================

print("=" * 60)
print("G-1 Reversal 시그널 생성")
print("=" * 60)

# ── 과거 N일 누적 수익률 ──────────────────────────────────────────
past_ret = ret_1d.rolling(window=REV_LOOKBACK).sum()

# ── 횡단면 z-score (급락 = 양수로 변환) ───────────────────────────
rev_signal = pd.DataFrame(index=past_ret.index, columns=past_ret.columns, dtype=float)

for date in past_ret.index:
    row = past_ret.loc[date].dropna()
    if len(row) < 50:
        continue
    z = (row - row.mean()) / row.std()
    rev_signal.loc[date, z.index] = -z  # 부호 반전: 급락 = 양수

print(f"rev_signal shape: {rev_signal.shape}")
print(f"유효 시작: {rev_signal.dropna(how='all').index.min().date()}")
print(f"일평균 유효 종목: {rev_signal.notna().sum(axis=1).mean():.0f}")
print(f"\n분포:\n{rev_signal.stack().describe().round(3)}")

G-1 Reversal 시그널 생성
rev_signal shape: (3178, 503)
유효 시작: 2013-06-25
일평균 유효 종목: 482

분포:
count    1532036.000
mean           0.000
std            0.999
min          -18.823
25%           -0.507
50%            0.009
75%            0.519
max           17.638
dtype: float64


In [4]:
# =============================================================================
# 셀 4 — G-1 백테스트 (전체 기간)
# =============================================================================
# 목적: 급락 종목 매수 → N일 보유 → 수익률 측정
# 산출물: g1_results
# =============================================================================

print("=" * 60)
print("G-1 Reversal 백테스트")
print("=" * 60)

def backtest_reversal(signal_df, ret_df, regime_df, top_n=30, hold_days=5):
    """
    Reversal 백테스트
    - 매일 시그널 상위 N종목 (급락) 매수
    - hold_days 후 수익률 측정
    """
    results = []
    dates = signal_df.index.tolist()
    
    for i, date in enumerate(dates):
        if i + hold_days >= len(dates):
            break
            
        sig = signal_df.loc[date].dropna()
        if len(sig) < top_n:
            continue
        
        # 상위 N종목 (급락 = 시그널 높음)
        top_tickers = sig.nlargest(top_n).index.tolist()
        
        # hold_days 후 날짜
        exit_date = dates[i + hold_days]
        
        # 보유 기간 수익률 (복리)
        hold_ret = (1 + ret_df.loc[dates[i+1]:exit_date, top_tickers]).prod() - 1
        port_ret = hold_ret.mean()
        
        # EW 벤치마크
        ew_hold = (1 + ret_df.loc[dates[i+1]:exit_date].mean(axis=1)).prod() - 1
        
        # 레짐
        reg = regime_df.loc[date, REGIME_COL] if date in regime_df.index else "Unknown"
        
        results.append({
            "entry_date": date,
            "exit_date": exit_date,
            "g1_ret": port_ret,
            "ew_ret": ew_hold,
            "regime": reg,
        })
    
    return pd.DataFrame(results)

# ── 백테스트 실행 ─────────────────────────────────────────────────
g1_results = backtest_reversal(rev_signal, ret_1d, regime_d, top_n=TOP_N, hold_days=REV_HOLD)
g1_results = g1_results.set_index("entry_date")

print(f"총 {len(g1_results)} 거래")
print(f"기간: {g1_results.index.min().date()} ~ {g1_results.index.max().date()}")

# ── 성과 계산 ─────────────────────────────────────────────────────
# 일별 수익률로 변환 (hold_days로 나눔)
g1_daily = g1_results["g1_ret"] / REV_HOLD
ew_daily = g1_results["ew_ret"] / REV_HOLD

g1_perf = calc_perf(g1_daily)
ew_perf = calc_perf(ew_daily)
t_stat = calc_tstat(g1_results["g1_ret"], g1_results["ew_ret"])

print(f"\n{'지표':12s} {'G-1 Rev':>12s} {'EW BM':>12s}")
print("-" * 40)
print(f"{'CAGR':12s} {g1_perf['CAGR']:>12.2%} {ew_perf['CAGR']:>12.2%}")
print(f"{'Sharpe':12s} {g1_perf['Sharpe']:>12.3f} {ew_perf['Sharpe']:>12.3f}")
print(f"{'MaxDD':12s} {g1_perf['MaxDD']:>12.2%} {ew_perf['MaxDD']:>12.2%}")
print(f"{'N trades':12s} {g1_perf['N']:>12d} {ew_perf['N']:>12d}")
print(f"\nt-stat vs EW: {t_stat:.3f}")

# ── 승률 ──────────────────────────────────────────────────────────
win_rate = (g1_results["g1_ret"] > g1_results["ew_ret"]).mean()
print(f"승률 (vs EW): {win_rate:.1%}")

G-1 Reversal 백테스트
총 3169 거래
기간: 2013-06-25 ~ 2026-01-29

지표                G-1 Rev        EW BM
----------------------------------------
CAGR               29.25%       18.96%
Sharpe              2.195        2.330
MaxDD             -47.69%      -33.34%
N trades             3169         3169

t-stat vs EW: 4.648
승률 (vs EW): 52.8%


In [5]:
# =============================================================================
# 셀 5 — G-1 레짐별 성과 분해
# =============================================================================

print("=" * 60)
print("G-1 Reversal — 레짐별 성과")
print("=" * 60)

print(f"\n{'레짐':22s} {'trades':>7s} {'G1 avg':>10s} {'EW avg':>10s} {'초과':>10s} {'t-stat':>8s} {'승률':>7s}")
print("-" * 85)

for reg in sorted(g1_results["regime"].unique()):
    mask = g1_results["regime"] == reg
    sub = g1_results.loc[mask]
    
    if len(sub) < 20:
        print(f"{reg:22s} {len(sub):>7d}   (샘플 부족)")
        continue
    
    g1_avg = sub["g1_ret"].mean()
    ew_avg = sub["ew_ret"].mean()
    excess = g1_avg - ew_avg
    t = calc_tstat(sub["g1_ret"], sub["ew_ret"])
    win = (sub["g1_ret"] > sub["ew_ret"]).mean()
    
    # 연율화 (5일 보유 → 연 50회)
    g1_ann = g1_avg * (252 / REV_HOLD)
    ew_ann = ew_avg * (252 / REV_HOLD)
    excess_ann = excess * (252 / REV_HOLD)
    
    print(f"{reg:22s} {len(sub):>7d} {g1_ann:>+10.2%} {ew_ann:>+10.2%} {excess_ann:>+10.2%} {t:>8.2f} {win:>7.1%}")

G-1 Reversal — 레짐별 성과

레짐                      trades     G1 avg     EW avg         초과   t-stat      승률
-------------------------------------------------------------------------------------
0_Neutral                  613    +40.30%    +27.05%    +13.25%     1.96   54.0%
1_Crash                     42   +315.68%   +137.31%   +178.37%     5.52   78.6%
2_Recovery_Early            60    +24.99%     +7.11%    +17.88%     1.61   58.3%
3_Contraction              147    +22.16%    +11.79%    +10.38%     1.04   52.4%
4_Recovery_Late            875    +21.62%    +15.16%     +6.46%     2.71   52.7%
5_Expansion               1330    +17.44%    +13.64%     +3.80%     1.88   52.0%
6_Peak                     102    -11.88%     +0.48%    -12.37%    -1.25   44.1%


In [6]:
# =============================================================================
# 셀 6 — Lookback × Hold 민감도 분석
# =============================================================================
# 목적: 최적 파라미터 탐색 + 과적합 여부 확인
# =============================================================================

print("=" * 60)
print("G-1 Lookback × Hold 민감도")
print("=" * 60)

lookbacks = [1, 3, 5, 10, 20]
holds = [1, 3, 5, 10, 20]

sensitivity_results = []

for lb in lookbacks:
    # 시그널 재생성
    past_ret = ret_1d.rolling(window=lb).sum()
    sig = pd.DataFrame(index=past_ret.index, columns=past_ret.columns, dtype=float)
    
    for date in past_ret.index:
        row = past_ret.loc[date].dropna()
        if len(row) < 50:
            continue
        z = (row - row.mean()) / row.std()
        sig.loc[date, z.index] = -z
    
    for hd in holds:
        # 백테스트
        bt = backtest_reversal(sig, ret_1d, regime_d, top_n=TOP_N, hold_days=hd)
        if len(bt) < 100:
            continue
        
        # 전체 성과
        t_all = calc_tstat(bt["g1_ret"], bt["ew_ret"])
        excess_all = (bt["g1_ret"].mean() - bt["ew_ret"].mean()) * (252 / hd)
        
        # Crash 성과
        crash_sub = bt[bt["regime"] == "1_Crash"]
        if len(crash_sub) >= 10:
            t_crash = calc_tstat(crash_sub["g1_ret"], crash_sub["ew_ret"])
            excess_crash = (crash_sub["g1_ret"].mean() - crash_sub["ew_ret"].mean()) * (252 / hd)
        else:
            t_crash = np.nan
            excess_crash = np.nan
        
        sensitivity_results.append({
            "lookback": lb,
            "hold": hd,
            "t_all": t_all,
            "excess_all": excess_all,
            "t_crash": t_crash,
            "excess_crash": excess_crash,
            "n_trades": len(bt),
        })

sens_df = pd.DataFrame(sensitivity_results)

# ── 피벗 테이블: t-stat (전체) ─────────────────────────────────────
print("\n[전체 기간 t-stat]")
pivot_t = sens_df.pivot(index="lookback", columns="hold", values="t_all").round(2)
print(pivot_t)

# ── 피벗 테이블: t-stat (Crash) ────────────────────────────────────
print("\n[Crash t-stat]")
pivot_crash = sens_df.pivot(index="lookback", columns="hold", values="t_crash").round(2)
print(pivot_crash)

# ── 최적 조합 ─────────────────────────────────────────────────────
best_all = sens_df.loc[sens_df["t_all"].idxmax()]
best_crash = sens_df.loc[sens_df["t_crash"].idxmax()]

print(f"\n★ 전체 최적: lookback={best_all['lookback']:.0f}, hold={best_all['hold']:.0f} (t={best_all['t_all']:.2f})")
print(f"★ Crash 최적: lookback={best_crash['lookback']:.0f}, hold={best_crash['hold']:.0f} (t={best_crash['t_crash']:.2f})")

G-1 Lookback × Hold 민감도

[전체 기간 t-stat]
hold        1     3     5     10    20
lookback                              
1         1.59  3.47  4.29  5.93  7.19
3         2.44  4.63  5.28  6.40  8.08
5         1.64  3.81  4.65  5.49  7.00
10        1.94  3.30  4.15  4.76  6.48
20        1.19  2.40  2.84  3.52  4.71

[Crash t-stat]
hold        1     3     5     10    20
lookback                              
1         0.56  2.13  3.59  3.93  3.33
3         0.59  3.30  4.58  4.45  4.01
5         2.11  4.70  5.52  5.24  4.18
10        1.87  2.96  2.98  2.09  2.72
20        1.39  2.50  2.50  2.59  2.78

★ 전체 최적: lookback=3, hold=20 (t=8.08)
★ Crash 최적: lookback=5, hold=5 (t=5.52)


In [7]:
# =============================================================================
# 셀 7 — G-1 최종 판정
# =============================================================================
#
# ─── G-1 Short-term Reversal 결과 요약 ───────────────────────────────────
#
#   [전체 기간]
#   - CAGR: 29.25% vs EW 18.96% → +10.29%p
#   - t-stat: 4.648 (강한 유의성)
#   - 승률: 52.8%
#
#   [레짐별]
#   | 레짐           | 초과(연율) | t-stat | 판정        |
#   |----------------|-----------|--------|-------------|
#   | 1_Crash        | +178.37%  | 5.52   | ✅ 핵심     |
#   | 0_Neutral      | +13.25%   | 1.96   | ✅ 유효     |
#   | 4_Recovery_Late| +6.51%    | 2.71   | ✅ 유효     |
#   | 3_Contraction  | +10.38%   | 1.04   | ⚠️ 약함     |
#   | 5_Expansion    | +3.88%    | 1.88   | ⚠️ 약함     |
#   | 6_Peak         | -12.37%   | -1.25  | ❌ 역방향   |
#
#   [민감도]
#   - 전체 최적: lookback=3, hold=20 (t=8.08)
#   - Crash 최적: lookback=5, hold=5 (t=5.52)
#   - 대부분 조합 t>2 → 과적합 아님
#
# ─── G-1 최종 판정 ───────────────────────────────────────────────────────
#
#   ✅ G-1 Short-term Reversal: 채택
#
#   운용 방식: 상시 운용 (레짐 판단 어려움 감안)
#   확정 파라미터: lookback=3, hold=10
#   예외: Peak 판단 시 비중 축소/OFF
#   보너스: Crash 감지 시 비중 확대
#
# ─── 팩터 현황 업데이트 ──────────────────────────────────────────────────
#
#   | 팩터 | 유형                | 유효 레짐              | 비고              |
#   |------|---------------------|------------------------|-------------------|
#   | A-3  | Value × Catalyst    | Neutral, Recovery_Late | LAG=45일          |
#   | D-3  | 변동성조정 모멘텀    | Neutral                | MOM_6_0/VOL       |
#   | D-1  | 단순 모멘텀          | Contraction            | MOM_12_1          |
#   | H-1  | 섹터 모멘텀          | 전 레짐                | 3M lookback       |
#   | F-1  | Piotroski F-score   | Bad (리스크 필터)      | 버전C, LAG=0      |
#   | P-7  | Net Stock Issuance  | 보조                   | N=50              |
#   | T-1a | Leader Spillover    | Expansion 가산         | σ=2.5, 7d         |
#   | T-1b | Leader Penalty      | Neutral 감점           | σ=2.5, 3~5d       |
#   | E-5  | Low Volatility      | Crash, Contraction     | 60일, 방어전용    |
#   | G-1  | Short-term Reversal | 상시 (Peak OFF)        | lb=3, hd=10       |
#
# ─── 레짐 커버리지 현황 ──────────────────────────────────────────────────
#
#   | 레짐           | 커버 팩터                     | 개수 |
#   |----------------|-------------------------------|------|
#   | Neutral        | A-3, D-3, T-1b, G-1           | 3.5  | ← 개선
#   | Recovery_Late  | A-3, H-1, G-1                 | 3    | ← 개선
#   | Expansion      | H-1, T-1a, G-1(약)            | 2    |
#   | Contraction    | D-1, F-1, E-5, G-1(약)        | 3    |
#   | Peak           | H-1, E-5                      | 1.5  |
#   | Crash          | F-1, E-5, G-1                 | 2.5  | ← 개선
#
# =============================================================================

print("G-1 Short-term Reversal: ✅ 채택")
print("운용: 상시 (Peak만 OFF)")
print("파라미터: lookback=3, hold=10")

G-1 Short-term Reversal: ✅ 채택
운용: 상시 (Peak만 OFF)
파라미터: lookback=3, hold=10
